In [2]:
%edit
%load_ext autoreload
%autoreload 2

IPython will make a temporary file named: /tmp/ipython_edit_d_8ikp4y/ipython_edit_60nu_kk2.py
The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [15]:
import torch
import pickle
import numpy as np

from spacegraph_codebase.data import PointSet, NeighborGraph,Point
from spacegraph_codebase.Place2Vec.cur_data_utils import load_pointset
from spacegraph_codebase.encoder import PointFeatureEncoder

In [7]:
data_dir = "data_collection/Place2Vec_center"
pointset, feature_embedding = load_pointset(data_dir)

In [8]:
a = pointset.pt_dict.keys()

In [9]:
pointset.pt_dict[100].data_mode

'training'

In [13]:
pointset.pt_dict[100].__dict__

{'id': 100,
 'coord': (-1701246.932032, 1622306.86305775),
 'coord_dim': 2,
 'features': (8, 138, 22),
 'data_mode': 'training'}

In [16]:
for mode in ["training", "validation", "test"]:
    coords = np.array([
        pt.coord for pt in pointset.pt_dict.values()
        if pt.data_mode == mode
    ])

    print(f"{mode}: {len(coords)} points")
    print(f"  x: {coords[:, 0].min():.2f} ~ {coords[:, 0].max():.2f}")
    print(f"  y: {coords[:, 1].min():.2f} ~ {coords[:, 1].max():.2f}")

training: 17460 points
  x: -1709999.84 ~ -1690007.32
  y: 1610000.29 ~ 1639997.24
validation: 2208 points
  x: -1709999.63 ~ -1690050.40
  y: 1610003.43 ~ 1639874.50
test: 2145 points
  x: -1709994.92 ~ -1690057.41
  y: 1610030.89 ~ 1639981.09


In [10]:
pointset.feature_embed_lookup([1,2])

tensor([[-0.1822, -0.0100,  0.0107, -0.0600, -0.0026, -0.0096,  0.0668,  0.1877,
          0.1172,  0.0866],
        [-0.0471,  0.1199,  0.1529, -0.1420, -0.0491, -0.0379,  0.0016,  0.1228,
          0.0791, -0.1165]], grad_fn=<EmbeddingBackward0>)

In [11]:
enc = PointFeatureEncoder(pointset.feature_embed_lookup, feature_embedding, pointset)

In [12]:
enc(list(range(1,5)))

tensor([[-0.2882,  0.1777,  0.0894, -0.1523,  0.0346, -0.0694,  0.5503,  0.7311,
          0.0200, -0.1022],
        [ 0.6731,  0.0568, -0.3007, -0.1678, -0.4136, -0.1912,  0.3823,  0.0565,
          0.2606, -0.0130],
        [ 0.0148, -0.2216, -0.1930,  0.4544,  0.1209, -0.3273,  0.3436,  0.0109,
         -0.6831,  0.0184],
        [-0.2503, -0.4166,  0.3098,  0.1840, -0.4797, -0.2017,  0.0681, -0.3577,
          0.2947, -0.3791]], grad_fn=<DivBackward0>)

In [ ]:
def make_data_samples(data_mode, pointset, data_dir, neighbor_tuple_path, neg_sample_num = 10):
    
    # print("Load PointSet....")
    # pointset = load_pointset(data_dir, point_data_path)

    print("Load {} neighbor_tuple_list".format(data_mode))
    neighbor_tuple_list = pickle.load(open(data_dir + "/" + neighbor_tuple_path, "rb"), encoding="latin1")
    # neighbor_tuple_list = pickle.load(open(data_dir + "/" + neighbor_tuple_path, "rb"))


    print("Do negative sampling and get NeighborGraph")
    ng_list = pointset.get_data_samples(neighbor_tuple_list, neg_sample_num, data_mode)
    
    return ng_list
#     print("Dump {} NeighborGraph()".format(data_mode))
#     pickle.dump([ng.serialize() for ng in ng_list], open(data_dir+"/neighborgraphs_{}.pkl".format(data_mode), "w"), protocol=pickle.HIGHEST_PROTOCOL)

In [38]:
data_mode = "test"
neighbor_tuple_path = "/neighbortuple_{}.pkl".format(data_mode)

In [40]:
ng_list = make_data_samples(data_mode, pointset, data_dir, neighbor_tuple_path)

Load test neighbor_tuple_list
Do negative sampling and get NeighborGraph


In [42]:
ng_list[100].serialize()

(25444,
 (16792, 24483, 13831, 18000, 18703, 16000, 12237, 13462, 3939, 5013),
 'test',
 [28078, 26189, 25736, 28072, 26121, 25843, 26645, 25573, 26878, 25487])